In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การแสดงแทน quantum computer สำหรับ Transpiler


<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันนี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

ในการแปลง abstract circuit ให้เป็น ISA circuit ที่สามารถรันบน QPU (quantum processing unit) เฉพาะได้ Transpiler ต้องการข้อมูลบางอย่างเกี่ยวกับ QPU ข้อมูลนี้พบได้ในสองที่: object `BackendV2` (หรือ `BackendV1` รุ่นเก่า) ที่คุณวางแผนส่ง job และ attribute `Target` ของ Backend

- [`Target`](../api/qiskit/qiskit.transpiler.Target) ประกอบด้วย constraint ที่เกี่ยวข้องทั้งหมดของ device เช่น native basis gate, qubit connectivity และข้อมูล pulse หรือ timing
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) มี `Target` โดยค่าเริ่มต้น ประกอบด้วยข้อมูลเพิ่มเติม เช่น [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap) และให้ interface สำหรับส่ง quantum circuit job

คุณยังสามารถระบุข้อมูลสำหรับ Transpiler ใช้อย่างชัดเจนได้ เช่น หากมี use case เฉพาะ หรือหากเชื่อว่าข้อมูลนี้จะช่วยให้ Transpiler สร้าง circuit ที่ optimize ได้ดีกว่า

ความแม่นยำที่ Transpiler สร้าง circuit ที่เหมาะสมที่สุดสำหรับ hardware เฉพาะขึ้นอยู่กับว่า `Target` หรือ `Backend` มีข้อมูลเกี่ยวกับ constraint มากแค่ไหน

> **Note:** เนื่องจากอัลกอริทึม transpilation พื้นฐานหลายตัวเป็น stochastic จึงไม่มีการรับประกันว่าจะหา circuit ที่ดีกว่า

หน้านี้แสดงตัวอย่างหลายรายการของการส่งข้อมูล QPU ไปยัง Transpiler ตัวอย่างเหล่านี้ใช้ target จาก mock backend [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke)

<span id="default-config"></span>
## การกำหนดค่าเริ่มต้น
การใช้ Transpiler แบบง่ายที่สุดคือการระบุข้อมูล QPU ทั้งหมดโดยให้ `Backend` หรือ `Target` เพื่อให้เข้าใจวิธีการทำงานของ Transpiler ให้สร้าง circuit และ transpile ด้วยข้อมูลต่าง ๆ ดังนี้

Import library ที่จำเป็นและสร้าง instance ของ QPU:
เพื่อแปลง abstract circuit ให้เป็น ISA circuit ที่รันบน processor เฉพาะได้ Transpiler ต้องการข้อมูลบางอย่างเกี่ยวกับ processor โดยทั่วไปข้อมูลนี้เก็บอยู่ใน [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) หรือ [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) ที่ส่งให้ Transpiler และไม่จำเป็นต้องใช้ข้อมูลเพิ่มเติม อย่างไรก็ตาม คุณยังสามารถระบุข้อมูลสำหรับ Transpiler ใช้อย่างชัดเจนได้ เช่น หากมี use case เฉพาะ หรือหากเชื่อว่าข้อมูลนี้จะช่วยให้ Transpiler สร้าง circuit ที่ optimize ได้ดีกว่า

หัวข้อนี้แสดงตัวอย่างหลายรายการของการส่งข้อมูลไปยัง Transpiler ตัวอย่างเหล่านี้ใช้ target จาก mock backend [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke)

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

ตัวอย่าง circuit ใช้ instance ของ [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) จาก circuit library ของ Qiskit

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

ตัวอย่างนี้ใช้การตั้งค่าเริ่มต้นเพื่อ transpile ไปยัง `target` ของ `backend` ซึ่งให้ข้อมูลทั้งหมดที่จำเป็นในการแปลง circuit ให้รันบน Backend ได้

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

ตัวอย่างนี้ใช้ในส่วนต่อมาของหัวข้อนี้เพื่อแสดงให้เห็นว่า coupling map และ basis gate เป็นข้อมูลสำคัญที่ต้องส่งไปยัง Transpiler เพื่อการสร้าง circuit ที่เหมาะสมที่สุด QPU มักสามารถเลือกการตั้งค่าเริ่มต้นสำหรับข้อมูลอื่นที่ไม่ได้ส่งไป เช่น timing และ scheduling

## Coupling map

coupling map คือกราฟที่แสดงว่า Qubit ใดบ้างที่เชื่อมต่อกันและมี two-qubit gate ระหว่างกัน บางครั้งกราฟนี้มีทิศทาง หมายความว่า two-qubit gate สามารถไปในทิศทางเดียวเท่านั้น อย่างไรก็ตาม Transpiler สามารถกลับทิศทางของ Gate ได้โดยเพิ่ม single-qubit gate เพิ่มเติม abstract quantum circuit สามารถแสดงแทนบนกราฟนี้ได้เสมอ แม้ว่า connectivity จะจำกัด โดยการแนะนำ SWAP gate เพื่อย้ายข้อมูล quantum ไปรอบ ๆ

Qubit จาก abstract circuit ของเราเรียกว่า _virtual Qubit_ และ Qubit บน coupling map เรียกว่า _physical Qubit_ Transpiler ให้การ mapping ระหว่าง virtual และ physical Qubit ขั้นตอนแรกของ transpilation ที่เรียกว่า _layout_ stage จะทำการ mapping นี้

> **Note:** แม้ว่า routing stage จะพัวพันกับ _layout_ stage ซึ่งเลือก Qubit จริง แต่โดยค่าเริ่มต้น หัวข้อนี้จะถือว่าเป็น stage แยกกันเพื่อความเรียบง่าย การรวม routing และ layout เรียกว่า _qubit mapping_ เรียนรู้เพิ่มเติมเกี่ยวกับ stage เหล่านี้ในหัวข้อ [Transpiler stages](transpiler-stages)

ส่ง keyword argument `coupling_map` เพื่อดูผลกระทบต่อ Transpiler:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

ดังที่แสดงข้างต้น มีการแทรก SWAP gate หลายตัว (แต่ละตัวประกอบด้วย CX gate สามตัว) ซึ่งจะทำให้เกิด error มากบน device ปัจจุบัน เพื่อดูว่า Qubit ใดถูกเลือกบน qubit topology จริง ให้ใช้ `plot_circuit_layout` จาก Qiskit Visualizations:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

ซึ่งแสดงว่า virtual Qubit 0-11 ของเรา ถูก map แบบตรงไปตรงมาไปยัง physical Qubit 0-11 ที่เรียงต่อกัน ลองกลับไปใช้ค่าเริ่มต้น (`optimization_level=1`) ซึ่งใช้ `VF2Layout` หากต้องการ routing

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

ตอนนี้ไม่มีการแทรก SWAP gate และ physical Qubit ที่เลือกเหมือนกับเมื่อใช้ class `target`

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

ตอนนี้ layout อยู่ในรูปวงแหวน เนื่องจาก layout นี้เคารพ connectivity ของ circuit จึงไม่มี SWAP gate ทำให้ได้ circuit ที่ดีกว่ามากสำหรับการประมวลผล

## Basis gate

quantum computer ทุกตัวรองรับ instruction set ที่จำกัด เรียกว่า _basis gate_ ทุก Gate ใน circuit ต้องถูกแปลเป็นส่วนประกอบของ set นี้ Set นี้ควรประกอบด้วย single- และ two-qubit gate ที่ให้ universal gate set หมายความว่าการดำเนินการ quantum ใด ๆ สามารถ decompose เป็น Gate เหล่านั้นได้ ซึ่งทำโดย [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator) และ basis gate สามารถระบุเป็น keyword argument ให้ Transpiler เพื่อให้ข้อมูลนี้

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

single-qubit gate เริ่มต้นบน _ibm_sherbrooke_ คือ `rz`, `x` และ `sx` และ two-qubit gate เริ่มต้นคือ `ecr` (echoed cross-resonance) CX gate ถูกสร้างจาก `ecr` gate ดังนั้นบน QPU บางตัว `ecr` ถูกระบุเป็น two-qubit basis gate ในขณะที่บางตัว `cx` เป็นค่าเริ่มต้น `ecr` gate คือส่วน _entangling_ ของ `cx` gate นอกจาก control gate แล้ว ยังมีคำสั่ง `delay` และ `measurement` ด้วย

> **Note:** QPU มี basis gate เริ่มต้น แต่คุณสามารถเลือก Gate ใดก็ได้ตามต้องการ ตราบใดที่ให้ instruction หรือเพิ่ม pulse gate (ดู [Create transpiler passes](custom-transpiler-pass)) basis gate เริ่มต้นคือ Gate ที่มีการทำ calibration บน QPU ไว้แล้ว จึงไม่จำเป็นต้องให้ instruction/pulse gate เพิ่มเติม ตัวอย่างเช่น บน QPU บางตัว `cx` คือ two-qubit gate เริ่มต้น และ `ecr` บางตัว ดู list ของ [native gate และ operation](/guides/qpu-information#native-gates) ที่เป็นไปได้สำหรับรายละเอียดเพิ่มเติม

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

สังเกตว่า object `CXGate` ถูก decompose เป็น `ecr` gate และ single-qubit basis gate
## อัตรา error ของ device
class `Target` สามารถประกอบด้วยข้อมูลเกี่ยวกับอัตรา error สำหรับการดำเนินการบน device
ตัวอย่างเช่น โค้ดต่อไปนี้ดึง properties ของ echoed cross-resonance (ECR) gate ระหว่าง Qubit 1 และ 0 (โปรดทราบว่า ECR gate มีทิศทาง):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

output แสดง duration ของ Gate (เป็นวินาที) และอัตรา error ของมัน เพื่อเปิดเผยข้อมูล error ให้ Transpiler สร้าง target model ด้วย `basis_gates` และ `coupling_map` จากข้างต้น และใส่ค่า error จาก Backend `FakeSherbrooke`